# JAX + JAM + NumPyro Workflow (GPU)

This notebook runs a clean end-to-end workflow:
1. Configure JAX/NumPyro backend and imports
2. Build light/mass ingredients and a JAM forward model
3. Fit `(thetaE, gamma, beta, q)` with NumPyro NUTS
4. Diagnose posterior and compare model vs observations


In [ ]:
# ---------- 1) Backend + imports ----------
import os
from pathlib import Path
from functools import partial

# Optional: choose one visible GPU
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
# Avoid inheriting stale platform filters
os.environ.pop('JAX_PLATFORMS', None)

import numpy as np
import matplotlib.pyplot as plt

import numpyro
numpyro.set_platform('gpu')

import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp

from numpyro.infer import NUTS, MCMC, init_to_median
import numpyro.distributions as dist
import arviz as az
import corner

from plotbin.plot_velfield import plot_velfield
from jam_axi_proj_jax import jam_axi_proj
from MGE_jax import MGE
from herculens.Util import param_util

# Local helper module in this folder
import importlib.util
sl_path = Path('SLCOSMO (1).py')
spec = importlib.util.spec_from_file_location('slcosmo_local', sl_path)
sl_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sl_module)
tool = sl_module.tool

print('JAX backend :', jax.default_backend())
print('JAX devices :', jax.devices())
if jax.default_backend() not in ('gpu', 'cuda', 'rocm'):
    raise RuntimeError('GPU backend is not active.')


In [ ]:
# ---------- 2) Build forward model ingredients ----------
import pickle as pkl

rng = np.random.default_rng(123)

# 2.1 Load lens light MGE from jackpot file
lens_light = pkl.load(open('lens_light_jackpot.pkl', 'rb'))[0]
surf_lum = jnp.asarray(lens_light['amp'])
sigma_lum = jnp.asarray(lens_light['sigma'])
_, qobs_lum = param_util.ellipticity2phi_q(lens_light['e1'], lens_light['e2'])
qobs_lum = jnp.asarray(qobs_lum)

# 2.2 Build a synthetic IFU coordinate set
xbin, ybin = rng.uniform(low=[-2, -2], high=[2, 2], size=[100, 2]).T

# 2.3 Cosmology + EPL mass decomposition
cosmology = {'Omegam': 0.3, 'Omegak': 0.0, 'w0': -1.0, 'wa': 0.0, 'h0': 70.0}
kwargs_epl = {'thetaE': 2.0, 'gamma': 2.0, 'zl': 0.2, 'zs': 0.4, 'cosmology': cosmology}
dl, ds, dls = tool.dldsdls(0.2, 0.4, cosmology, n=20)
distance = float(dl[0])

mge_epl = MGE(
    tool.EPL_msunmpc,
    'thetaE',
    n_gauss=10,
    n_terms=28,
    sigma_start_mult=1/100,
    sigma_end_mult=50,
)
mass_amp_ref, mass_sig_ref = mge_epl.decompose(**kwargs_epl)

# 2.4 JAM fixed settings
inc = 90.0
mbh = 0.0
pixsize = 0.8
sigmapsf = jnp.array([0.6, 1.2])
normpsf = jnp.array([0.7, 0.3])

jam_obj = jam_axi_proj()

# A partially bound forward call; only mass + beta + qobs_pot vary in inference
jam_eval = partial(
    jam_obj.get_kinematics,
    surf_lum=surf_lum,
    sigma_lum=sigma_lum,
    qobs_lum=qobs_lum,
    inc=inc,
    mbh=mbh,
    distance=distance,
    xbin=xbin,
    ybin=ybin,
    sigmapsf=sigmapsf,
    normpsf=normpsf,
    pixsize=pixsize,
    logistic=False,
    step=0.05,
    moment='zz',
    align='cyl',
    ml=None,
    quiet=True,
)

# Reference model (used to build synthetic observation)
beta_ref = jnp.zeros_like(surf_lum)
qobs_pot_ref = jnp.full_like(mass_amp_ref, 0.9)
model_ref, *_ = jam_eval(
    surf_pot=mass_amp_ref,
    sigma_pot=mass_sig_ref,
    beta=beta_ref,
    qobs_pot=qobs_pot_ref,
)

plot_velfield(xbin, ybin, jnp.clip(model_ref, -1e3, 1e3))
plt.title('Reference JAM model')
plt.show()


In [ ]:
# ---------- 3) Inference with NumPyro ----------
key = jax.random.PRNGKey(42)
err_frac = 0.20
# Synthetic observed velocity dispersion with 20% relative noise
obs_sigma = err_frac * jnp.clip(jnp.abs(model_ref), 1e-6, None)
noise = obs_sigma * jax.random.normal(key, shape=model_ref.shape)
sigma_obs = model_ref + noise
def lens_kinematics_model(sigma_obs):
    # Priors
    thetaE = numpyro.sample('thetaE', dist.Uniform(1.9, 2.1))
    gamma = numpyro.sample('gamma', dist.Uniform(1.6, 2.4))
    beta = numpyro.sample('beta', dist.Uniform(-0.5, 0.8))
    q = numpyro.sample('q', dist.Uniform(0.3, 1.0))
    beta_vec = jnp.full_like(surf_lum, beta)
    mass_amp, mass_sig = mge_epl.decompose(
        thetaE=thetaE,
        gamma=gamma,
        zl=kwargs_epl['zl'],
        zs=kwargs_epl['zs'],
        cosmology=kwargs_epl['cosmology'],
    )
    qobs_pot_local = jnp.full_like(mass_amp, q)
    pred_model, *_ = jam_eval(
        surf_pot=mass_amp,
        sigma_pot=mass_sig,
        beta=beta_vec,
        qobs_pot=qobs_pot_local,
    )
    sigma = err_frac * jnp.clip(jnp.abs(pred_model), 1e-6, None)
    with numpyro.plate('pixels', pred_model.size):
        numpyro.sample('obs', dist.Normal(pred_model, sigma), obs=sigma_obs)
nuts = NUTS(lens_kinematics_model, init_strategy=init_to_median(), target_accept_prob=0.8)
mcmc = MCMC(
    nuts,
    num_warmup=500,
    num_samples=1500,
    num_chains=4,
    chain_method='vectorized',
    progress_bar=True,
)
mcmc.run(jax.random.PRNGKey(0), sigma_obs=sigma_obs)


In [ ]:
# ---------- 4) Posterior diagnostics ----------
inf_data = az.from_numpyro(mcmc)
div = inf_data.sample_stats.diverging.values.sum(axis=1).T
print('divergences per chain per step:\n', div)
varnames = ['thetaE', 'gamma', 'beta', 'q']
display(az.summary(inf_data, var_names=varnames))
_ = az.plot_trace(inf_data, var_names=varnames)
plt.tight_layout()
plt.show()
_ = corner.corner(inf_data, var_names=varnames, truths=[2.0, 2.0, 0.0, 0.9])
plt.show()


In [ ]:
# ---------- 5) Posterior median prediction ----------
median = inf_data.posterior.median(dim=('draw', 'chain'))

thetaE_med = float(np.array(median.thetaE))
gamma_med = float(np.array(median.gamma))
beta_med = float(np.array(median.beta))
q_med = float(np.array(median.q))

mass_amp_med, mass_sig_med = mge_epl.decompose(
    thetaE=thetaE_med,
    gamma=gamma_med,
    zl=kwargs_epl['zl'],
    zs=kwargs_epl['zs'],
    cosmology=kwargs_epl['cosmology'],
)

qobs_pot_med = jnp.full_like(mass_amp_med, q_med)
beta_vec_med = jnp.full_like(surf_lum, beta_med)

pred_model, *_ = jam_eval(
    surf_pot=mass_amp_med,
    sigma_pot=mass_sig_med,
    beta=beta_vec_med,
    qobs_pot=qobs_pot_med,
)

print('Posterior medians:')
print({'thetaE': thetaE_med, 'gamma': gamma_med, 'beta': beta_med, 'q': q_med})


In [ ]:
# ---------- 6) Radial comparison: observed vs model ----------
r = np.hypot(xbin, ybin).ravel()
y_obs = np.array(sigma_obs).ravel()
y_mod = np.array(pred_model).ravel()
y_err = np.array(obs_sigma).ravel()
plt.figure(figsize=(12, 4))
plt.errorbar(
    r, y_obs, yerr=y_err,
    fmt='o', ms=2.5, lw=0.8, elinewidth=0.6, capsize=2, alpha=0.6,
    label=r'$\sigma_{\rm obs}$ with 20% errors',
)
plt.scatter(r, y_mod, s=12, c='red', alpha=0.9, zorder=3, label='posterior median model')
plt.xlabel('radius (same units as xbin/ybin)')
plt.ylabel(r'$\sigma$')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
